# Batch Error Diagnosis

**Last updated:** 2026-04-15 21:50 PT

**Track:** Learning | **Sub-Ability:** Observational learning

Given a multi-error build/test output, can the model identify ALL distinct root causes in one pass — not just the first one?

The real-world failure mode: agents fix one error, re-run, see the next, fix it, re-run... doing N cycles instead of 1. Observed in live agent runs: 18 deploy cycles for a single feature; 4 sequential build cycles where a single enumeration would have sufficed.

This benchmark teaches (via worked examples) the pattern of enumerating ALL distinct root causes before making any change, then scores whether the model transfers that pattern to a new multi-error build output.

**Difficulty grid:** complexity (independent / chained / redundant) x evidence (3 / 4 / 5 examples) x 3 seeds = 27 items

- `independent_errors` — errors don't affect each other
- `chained_errors` — fixing one would expose another; count is still N distinct root causes
- `redundant_errors` — multiple symptoms of the same root cause; model must not over-count

**Scoring:** The model outputs an integer count of distinct root causes. Exact match against ground truth.

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import re
import json
import time

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)

In [ ]:
def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()

def extract_number(response: str) -> str:
    """Extract a number from model response.

    Strategy: strip thinking + markdown, look for the last line that is just a
    number (the final-answer convention), otherwise fall back to the last number
    anywhere in the text.
    """
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    for line in reversed(lines):
        clean = line.strip('.!?, ')
        if re.match(r'^-?\d+$', clean):
            return clean
    nums = re.findall(r'-?\d+', text)
    return nums[-1] if nums else ''

In [ ]:
# Constants used in results analysis
COMPLEXITY_LEVELS = ['independent_errors', 'chained_errors', 'redundant_errors']
EVIDENCE_LEVELS = ['few', 'mid', 'many']
SEEDS = 3

In [ ]:
# === Load dataset from Kaggle ===
import os, glob
candidates = glob.glob('/kaggle/input/**/batch_error_diagnosis_dataset.csv', recursive=True)
if candidates:
    csv_path = candidates[0]
else:
    csv_path = '/kaggle/input/batch_error_diagnosis_dataset.csv'
print(f'Loading from: {csv_path}')
dataset = pd.read_csv(csv_path)
print(f'Dataset: {len(dataset)} items')

In [ ]:
# === Prompt templates ===

PRE_P = """You are shown worked examples of diagnosing multi-error build/test outputs.
Each example shows a build output containing multiple error lines and a full enumeration of distinct root causes.

{material}

Now apply the same diagnostic approach to this new build output:

{test_input}

Enumerate every distinct root cause, then give the count as a single integer on the last line."""

STUDY_P = """You are shown worked examples of diagnosing multi-error build/test outputs.
Each example shows a build output containing multiple error lines and a full enumeration of distinct root causes.

{material}

Your task:
1. Analyze these examples carefully.
2. Write down the EXACT diagnostic procedure being followed.
3. Be precise about: (a) how to enumerate every distinct error line, (b) how to identify which lines share a root cause (so you don't over-count), and (c) how to handle chained/cascading errors (count each distinct root cause, not the chain length).
4. Verify your understanding by mentally re-running the procedure on at least one example.

Write clear, detailed notes that would let someone else reproduce this diagnostic procedure exactly."""

POST_P = """You previously studied a diagnostic procedure from worked examples and wrote these notes:

--- YOUR NOTES ---
{notes}
--- END NOTES ---

Here are the original worked examples for reference:
{material}

Apply the diagnostic procedure to this new build output:

{test_input}

Enumerate every distinct root cause, then give the count as a single integer on the last line."""

## Evaluation

In [ ]:
@kbench.task(name='batch_error_diagnosis',
             description='Pre/post learning: diagnose ALL distinct root causes in a multi-error build output, not just the first one')
def batch_error_diagnosis(llm) -> float:
    model_name = str(llm)
    correct = 0
    total = len(dataset)
    all_results = []

    for _, row in dataset.iterrows():
        material = row['material']
        test_input = row['test_input']
        expected = row['expected']
        complexity = row['complexity']
        evidence = row['evidence']
        n_examples = int(row['n_examples'])
        difficulty_label = row['difficulty_label']
        seed = int(row['seed'])
        item_desc = row['item_desc']
        ts = time.strftime('%Y-%m-%d %H:%M:%S')
        task_id = f'{complexity}_{evidence}_{seed}'

        t0 = time.time()
        pre_raw = llm.prompt(PRE_P.format(material=material, test_input=test_input))
        pre_time = time.time() - t0
        pre_extracted = extract_number(pre_raw)
        pre_correct = pre_extracted == str(expected)

        t0 = time.time()
        study_raw = llm.prompt(STUDY_P.format(material=material))
        study_time = time.time() - t0
        notes = strip_thinking(study_raw)

        t0 = time.time()
        post_raw = llm.prompt(POST_P.format(notes=notes, material=material, test_input=test_input))
        post_time = time.time() - t0
        post_extracted = extract_number(post_raw)
        post_correct = post_extracted == str(expected)

        if post_correct:
            correct += 1

        all_results.append({
            'timestamp': ts, 'model': model_name, 'task_id': task_id,
            'complexity': complexity, 'evidence': evidence, 'n_examples': n_examples,
            'difficulty_label': difficulty_label, 'seed': seed, 'item_desc': item_desc,
            'test_input': test_input, 'expected': str(expected),
            'pre_raw': pre_raw, 'pre_extracted': pre_extracted, 'pre_correct': int(pre_correct),
            'study_notes': notes,
            'post_raw': post_raw, 'post_extracted': post_extracted, 'post_correct': int(post_correct),
            'pre_time_s': pre_time, 'study_time_s': study_time, 'post_time_s': post_time,
        })

        b = 'Y' if pre_correct else 'N'
        l = 'Y' if post_correct else 'N'
        print(f'[{model_name:40s}] [{difficulty_label:28s}] expected={expected!s:>3s}  '
              f'pre="{pre_extracted}"({b})  post="{post_extracted}"({l})')
        kbench.assertions.assert_equal(post_extracted, str(expected))

    score = correct / total
    print(f'\nScore: {correct}/{total} = {score:.4f}')
    return score

try:
    runs = batch_error_diagnosis.evaluate(llm=[kbench.llm],
                                          n_jobs=1, timeout=3600, max_attempts=1)
    print(f'\nDone: {len(runs.as_dataframe())} items')
except Exception as e:
    print(f'SDK post-processing error (non-fatal): {e}')

## Results

In [ ]:
results = pd.DataFrame(all_results)
print(f'Total runs: {len(results)}\n')

# Per-model summary
print('SCALING CHECK — Pre-learning vs post-learning accuracy by model:')
print('=' * 70)
for model in sorted(results['model'].unique()):
    sub = results[results['model'] == model]
    pre = sub['pre_correct'].mean()
    post = sub['post_correct'].mean()
    gain = post - pre
    print(f'  {str(model):45s}  pre={pre:.1%}  post={post:.1%}  gain={gain:+.1%}  (n={len(sub)})')

# Per model x complexity
print()
print('By model x complexity (PRE):')
print('-' * 70)
for model in sorted(results['model'].unique()):
    row = f'  {str(model):45s}'
    for c in COMPLEXITY_LEVELS:
        sub = results[(results['model'] == model) & (results['complexity'] == c)]
        if len(sub) > 0:
            row += f'  {c[:4]}={sub["pre_correct"].mean():.0%}'
    print(row)

# Per model x evidence
print()
print('By model x evidence (PRE):')
print('-' * 70)
for model in sorted(results['model'].unique()):
    row = f'  {str(model):45s}'
    for ev in EVIDENCE_LEVELS:
        sub = results[(results['model'] == model) & (results['evidence'] == ev)]
        if len(sub) > 0:
            row += f'  {ev}={sub["pre_correct"].mean():.0%}'
    print(row)

print()
print(f'Timing: pre={results["pre_time_s"].mean():.1f}s  study={results["study_time_s"].mean():.1f}s  post={results["post_time_s"].mean():.1f}s')

In [ ]:
print('STUDY NOTES INSPECTION')
print('=' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    print(f'\n--- [{row["difficulty_label"]}] seed={row["seed"]} ---')
    print(f'Item: {row["item_desc"]}')
    print(f'Expected: {row["expected"]}')
    print(f'Pre: "{row["pre_extracted"]}" ({b})  Post: "{row["post_extracted"]}" ({l})')
    print(f'Notes (first 300 chars): {str(row["study_notes"])[:300]}')
    print('...' if len(str(row['study_notes'])) > 300 else '')

In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

labels = sorted(results['difficulty_label'].unique())
pre_scores = [results[results['difficulty_label']==l]['pre_correct'].mean() for l in labels]
post_scores = [results[results['difficulty_label']==l]['post_correct'].mean() for l in labels]

x = range(len(labels))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
ax1.bar([i-width/2 for i in x], pre_scores, width, label='Pre-learning', color='#4C72B0')
ax1.bar([i+width/2 for i in x], post_scores, width, label='Post-learning', color='#55A868')
ax1.set_ylabel('Accuracy')
ax1.set_title('Batch Error Diagnosis: Pre vs Post-Learning')
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [p-b for b,p in zip(pre_scores, post_scores)]
colors = ['#55A868' if g>=0 else '#C44E52' for g in gains]
ax2.bar(range(len(labels)), gains, color=colors)
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty')
ax2.set_xticks(list(x))
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('batch_error_diagnosis_results.png', dpi=150, bbox_inches='tight')
plt.show()